In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import urllib.request
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from pathlib import Path

## Config

In [ ]:
DATA_ROOT = Path("../Datasets/kaggle_narutohandsigns")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

SEAL_CLASSES = ["bird", "boar", "dog", "dragon", "hare", "horse", "monkey", "ox", "ram", "rat", "snake", "tiger", "zero"]

In [ ]:
# sanity check to see if folder actually exists and contains images
for split, path in [("Train", TRAIN_DIR), ("Test", TEST_DIR)]:
    print(f"\n{split}:")
    for cls in SEAL_CLASSES:
        folder = path / cls
        if folder.exists():
            count = len(list(folder.glob("*.jpg")) + list(folder.glob("*.png")))
            print(f"  {cls:<10} {count} images")
        else:
            print(f"  {cls:<10} NOT FOUND")

## Landmark Extraction

In [ ]:
model_path = "hand_landmarker.task"

if not Path(model_path).exists():
    print("Downloading hand landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        model_path
    )
    print("Done.")
else:
    print("Model already downloaded.")

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model_path),
    num_hands=1
)
landmarker = vision.HandLandmarker.create_from_options(options)

In [26]:
from skimage.feature import hog
from skimage.color import rgb2gray
import cv2

def extract_hog(image_path: Path, img_size=(64, 64)) -> np.ndarray | None:
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, img_size)
    features = hog(img, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), visualize=False)
    return features

## Array Builder

In [27]:
from tqdm import tqdm

def load_split(split_dir: Path, max_per_class: int = 300):
    X, y = [], []
    failed = 0
    for cls in SEAL_CLASSES:
        images = list((split_dir / cls).glob("*.jpg")) + \
                 list((split_dir / cls).glob("*.png"))
        images = images[:max_per_class]
        for img_path in tqdm(images, desc=cls, leave=False):
            vec = extract_hog(img_path)
            if vec is not None:
                X.append(vec)
                y.append(cls)
            else:
                failed += 1
    print(f"Loaded {len(X)} samples from {split_dir.name}, {failed} failed")
    return np.array(X), np.array(y)

X_train, y_train = load_split(TRAIN_DIR)
X_test,  y_test  = load_split(TEST_DIR)

monkey:  56%|█████▌    | 76/136 [00:41<00:58,  1.02it/s]  E0000 00:00:1777370133.522198  120976 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-04-28T12:00:33.488993+02:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
monkey:  87%|████████▋ | 118/136 [01:42<00:24,  1.35s/it]E0000 00:00:1777370193.543035  120976 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-04-28T12:00:33.488993+02:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
ox:  14%|█▍        | 24/169 [00:33<03:13,  1.33s/it]     E0000 00:00:1777370253.546529  120976 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-04-28T12:00:33.488993+02:00
=== Source Location Trace: ===
wireless/android/play/playlog/cp

KeyboardInterrupt: 

## Class Dist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, y, title in [(axes[0], y_train, "Train"), (axes[1], y_test, "Test")]:
    classes, counts = np.unique(y, return_counts=True)
    ax.bar(classes, counts, color="steelblue")
    ax.set_title(f"{title} class distribution")
    ax.set_xlabel("Seal")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Sanity Check

In [ ]:
sample_path = next((TRAIN_DIR / "Dragon").glob("*.jpg"))
img = cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB)

with mp_hands.Hands(static_image_mode=True, max_num_hands=1) as hands:
    result = hands.process(img)

annotated = img.copy()
if result.multi_hand_landmarks:
    mp.solutions.drawing_utils.draw_landmarks(
        annotated,
        result.multi_hand_landmarks[0],
        mp_hands.HAND_CONNECTIONS
    )

plt.figure(figsize=(5, 5))
plt.imshow(annotated)
plt.title("Sample: Dragon — MediaPipe landmarks")
plt.axis("off")
plt.show()

## Training

In [ ]:
le = LabelEncoder().fit(SEAL_CLASSES)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",    SVC(kernel="rbf", C=10, gamma="scale", probability=True))
])

pipe.fit(X_train, y_train_enc)

train_acc = accuracy_score(y_train_enc, pipe.predict(X_train))
test_acc  = accuracy_score(y_test_enc,  pipe.predict(X_test))
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test  accuracy: {test_acc:.3f}")

In [ ]:
y_pred = pipe.predict(X_test_enc if False else X_test)  # already encoded above
y_pred = pipe.predict(X_test)

cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(11, 9))
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
plt.title("Confusion Matrix — MediaPipe + SVM")
plt.tight_layout()
plt.show()

In [ ]:
proba = pipe.predict_proba(X_test)
top3_correct = sum(
    y_test_enc[i] in np.argsort(proba[i])[-3:]
    for i in range(len(y_test_enc))
)
print(f"Top-3 accuracy: {top3_correct / len(y_test_enc):.3f}")

## Live Demo

In [ ]:
cap = cv2.VideoCapture(0)
print("Press Q to quit")

with mp_hands.Hands(static_image_mode=False, max_num_hands=1) as hands:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        label = "No hand detected"
        
        if result.multi_hand_landmarks:
            vec = np.array([[p.x, p.y, p.z]
                            for p in result.multi_hand_landmarks[0].landmark]).flatten()
            pred_enc = pipe.predict([vec])[0]
            label = le.inverse_transform([pred_enc])[0]
        
        cv2.putText(frame, label, (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 200, 255), 3)
        cv2.imshow("PoC — Naruto Handsign", frame)
        
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()